# Day 8 Validation Runner
Purpose: Validate the `day8-final-compliance-hardening` branch directly in Google Colab using the Google Drive `Data/` bundle.

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
%%bash
# Cell 2: Clone Day 8 branch
rm -rf /content/AIGC_Fake_Detection
git clone -b day8-final-compliance-hardening https://github.com/IanJ332/AIGC_Fake_Detection.git /content/AIGC_Fake_Detection
cd /content/AIGC_Fake_Detection
git branch --show-current
git log --oneline -3

In [ ]:
%%bash
# Cell 3: Install dependencies
cd /content/AIGC_Fake_Detection
pip install -r requirements.txt

In [ ]:
# Cell 4: Verify Drive Data
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/AIGC/Data")

required = [
    DATA_DIR / "extracted" / "entities.csv",
    DATA_DIR / "extracted" / "result_tuples.csv",
    DATA_DIR / "extracted" / "paper_entity_summary.csv",
    DATA_DIR / "extracted" / "paper_section_stats.csv",
    DATA_DIR / "extracted" / "extraction_registry.csv",
    DATA_DIR / "sections" / "sections.jsonl",
    DATA_DIR / "tables" / "table_candidates.jsonl",
    DATA_DIR / "registry" / "manifest_100.csv",
    DATA_DIR / "index" / "research_corpus.duckdb",
]

missing = []
for p in required:
    exists = p.exists()
    print(p, "=>", exists)
    if not exists:
        missing.append(str(p))

assert not missing, "Missing required Drive Data files:\n" + "\n".join(missing)
print("READY_TO_RUN_DAY8_ON_DRIVE")

In [ ]:
%%bash
# Cell 5: Compile checks
cd /content/AIGC_Fake_Detection
python -m py_compile src/query/*.py src/extract/*.py src/parse/*.py eval/*.py scripts/*.py
python -m compileall src/query src/extract src/parse eval

In [ ]:
%%bash
# Cell 6: Eval-only reproduction
cd /content/AIGC_Fake_Detection
bash scripts/reproduce_all.sh /content/drive/MyDrive/AIGC/Data --eval-only

In [ ]:
%%bash
# Cell 7: Budget eval
cd /content/AIGC_Fake_Detection
python eval/run_budget_eval.py \
  --data-dir /content/drive/MyDrive/AIGC/Data \
  --questions eval/questions_40.jsonl \
  --out-dir artifacts/reports

In [ ]:
%%bash
# Cell 8: Critical demo questions (with output capture)
cd /content/AIGC_Fake_Detection
export PYTHONPATH=/content/AIGC_Fake_Detection:$PYTHONPATH

echo "===== Citation Graph Demo ====="
python -m src.query.cli \
  --data-dir /content/drive/MyDrive/AIGC/Data \
  --question "Which papers in the corpus build on the GenImage benchmark?" | tee /content/demo_citation.txt

echo "===== Multihop Demo ====="
python -m src.query.cli \
  --data-dir /content/drive/MyDrive/AIGC/Data \
  --question "Find papers that use both the CLIP model and the Accuracy metric." | tee /content/demo_multihop.txt

echo "===== Quantitative Demo ====="
python -m src.query.cli \
  --data-dir /content/drive/MyDrive/AIGC/Data \
  --question "How many papers mention GenImage?" | tee /content/demo_quant.txt

In [ ]:
%%bash
# Cell 9: Sync validation outputs to Drive
mkdir -p /content/drive/MyDrive/AIGC/Data/reports/day8_eval

cp -r /content/AIGC_Fake_Detection/eval/results/* \
  /content/drive/MyDrive/AIGC/Data/reports/day8_eval/ 2>/dev/null || true

cp /content/AIGC_Fake_Detection/artifacts/reports/budget_eval_results.csv \
  /content/drive/MyDrive/AIGC/Data/reports/day8_eval/ 2>/dev/null || true

cp /content/AIGC_Fake_Detection/artifacts/reports/budget_eval_summary.md \
  /content/drive/MyDrive/AIGC/Data/reports/day8_eval/ 2>/dev/null || true

echo "Saved Day 8 validation outputs to Drive:"
ls -lah /content/drive/MyDrive/AIGC/Data/reports/day8_eval

In [ ]:
# Cell 10: Final status summary (with semantic validation)
from pathlib import Path

drive_report_dir = Path("/content/drive/MyDrive/AIGC/Data/reports/day8_eval")
repo_report_dir = Path("/content/AIGC_Fake_Detection/artifacts/reports")

checks = {
    "day6_eval_summary": (drive_report_dir / "day6_eval_summary.md").exists() or (repo_report_dir / "day6_eval_summary.md").exists(),
    "budget_eval_summary": (drive_report_dir / "budget_eval_summary.md").exists() or (repo_report_dir / "budget_eval_summary.md").exists(),
    "budget_eval_results": (drive_report_dir / "budget_eval_results.csv").exists() or (repo_report_dir / "budget_eval_results.csv").exists(),
}

# Semantic Checks
def check_file(path, required_str, forbidden_str=None):
    if not Path(path).exists(): return False
    content = Path(path).read_text()
    if required_str not in content: return False
    if forbidden_str and forbidden_str in content: return False
    return True

checks["semantic_citation"] = check_file("/content/demo_citation.txt", "TIER: citation_graph", "Top 10 dataset entities")
checks["semantic_multihop"] = check_file("/content/demo_multihop.txt", "TIER: multihop")
checks["semantic_quantitative"] = check_file("/content/demo_quant.txt", "Papers mentioning 'GenImage'")

for k, v in checks.items():
    print(f"{k:20} => {v}")

if all(checks.values()):
    print("\nRESULT: DAY8_VALIDATION_PASS")
else:
    print("\nRESULT: DAY8_VALIDATION_CAUTION")